# Modern Hadoop Stack - Notebook Demo

One notebook, every engine in the stack: Spark on YARN, Python MapReduce
via `mrjob`, Hive-on-Tez via `pyhive`, and HBase via `happybase`, plus a
side-by-side comparison of the two wordcount implementations at the end.

## Cluster status

In [1]:
import pandas as pd
import requests

nodes = requests.get("http://hadoop-master:8088/ws/v1/cluster/nodes", timeout=5).json()["nodes"].get("node", [])
pd.DataFrame(
    [{"node": n["nodeHostName"], "state": n["state"], "containers": n.get("numContainers", 0)} for n in nodes]
)

,node,state,containers
0,hadoop-worker2,RUNNING,0
1,hadoop-worker1,RUNNING,0


## Spark on YARN

In [2]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, explode, split

spark = SparkSession.builder.appName("jupyter-demo").master("yarn").getOrCreate()

words = spark.read.text("/user/root/input").select(explode(split(col("value"), r"\s+")).alias("word"))
spark_counts = words.groupBy("word").count().orderBy(col("count").desc())
spark_counts_pd = spark_counts.toPandas()

# Stop the session immediately once we've pulled results into pandas --
# an interactive SparkSession left running holds its YARN executors for
# the rest of the notebook's life, starving every job after it on this
# tiny 4GB-total cluster.
spark.stop()

spark_counts_pd

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


,word,count
0,hello,2
1,hadoop,2
2,world,1
3,modern,1


## MapReduce - Python via `mrjob`

Same MapReduce framework, same YARN cluster, written in Python instead of
Java via Hadoop Streaming.

In [3]:
import json
import os
import subprocess

subprocess.run(["hdfs", "dfs", "-rm", "-r", "-f", "/user/root/notebook-mrjob-output"], check=True)
subprocess.run(
    [
        "python3", "/jobs/mrjob/wordcount.py", "-r", "hadoop",
        "hdfs:///user/root/input/sample.txt",
        "--hadoop-streaming-jar", os.environ["HADOOP_STREAMING_JAR"],
        "-o", "hdfs:///user/root/notebook-mrjob-output",
    ],
    check=True,
)

output = subprocess.run(
    ["bash", "-c", "hdfs dfs -cat /user/root/notebook-mrjob-output/part-*"],
    check=True, capture_output=True, text=True,
).stdout

# mrjob's default protocol JSON-encodes both the key and value per line.
rows = []
for line in output.strip().splitlines():
    word_json, count = line.split("\t")
    rows.append({"word": json.loads(word_json), "count": int(count)})
mrjob_counts_pd = pd.DataFrame(rows)
mrjob_counts_pd

SLF4J: Class path contains multiple SLF4J bindings.
SLF4J: Found binding in [jar:file:/opt/hadoop/share/hadoop/common/lib/slf4j-reload4j-1.7.36.jar!/org/slf4j/impl/StaticLoggerBinder.class]
SLF4J: Found binding in [jar:file:/opt/tez/lib/slf4j-reload4j-1.7.36.jar!/org/slf4j/impl/StaticLoggerBinder.class]
SLF4J: See http://www.slf4j.org/codes.html#multiple_bindings for an explanation.
SLF4J: Actual binding is of type [org.slf4j.impl.Reload4jLoggerFactory]


Deleted /user/root/notebook-mrjob-output


No configs found; falling back on auto-configuration
No configs specified for hadoop runner
Looking for hadoop binary in /opt/hadoop/bin...
Found hadoop binary: /opt/hadoop/bin/hadoop


Using Hadoop version 3.4.3
Creating temp directory /tmp/wordcount.root.20260922.082351.792538


uploading working dir files to hdfs:///user/root/tmp/mrjob/wordcount.root.20260922.082351.792538/files/wd...


...

   map 0% reduce 0%


   map 100% reduce 0%
   map 100% reduce 100%
  Job job_1790059083631_0046 completed successfully


  Output directory: hdfs:///user/root/notebook-mrjob-output
Counters: 54
	File Input Format Counters 
		Bytes Read=59
	File Output Format Counters 
		Bytes Written=42
	File System Counters
		FILE: Number of bytes read=81
		FILE: Number of bytes written=946520
		FILE: Number of large read operations=0
		FILE: Number of read operations=0
		FILE: Number of write operations=0
		HDFS: Number of bytes read=267
		HDFS: Number of bytes read erasure-coded=0
		HDFS: Number of bytes written=42
		HDFS: Number of large read operations=0
		HDFS: Number of read operations=11
		HDFS: Number of write operations=2
	Job Counters 
		Data-local map tasks=2
		Launched map tasks=2
		Launched reduce tasks=1
		Total megabyte-milliseconds taken by all map tasks=3675136
		Total megabyte-milliseconds taken by all reduce tasks=1636352
		Total time spent by all map tasks (ms)=3589
		Total time spent by all maps in occupied slots (ms)=3589
		Total time spent by all reduce tasks (ms)=1598
		Total time spent by all re

Removing temp directory /tmp/wordcount.root.20260922.082351.792538...


,word,count
0,hadoop,2
1,hello,2
2,modern,1
3,world,1


## Hive via pyhive

In [4]:
from pyhive import hive

conn = hive.Connection(host="hiveserver2", port=10000, username="root", auth="NONE")
cursor = conn.cursor()
cursor.execute("SELECT * FROM people")
columns = [d[0].split(".")[-1] for d in cursor.description]
hive_pd = pd.DataFrame(cursor.fetchall(), columns=columns)
hive_pd

,name,age,department,city
0,Alice,30,Engineering,Berlin
1,Bob,25,Engineering,Munich
2,Carla,34,Sales,Berlin
3,David,45,Sales,Hamburg
4,Elena,29,Marketing,Munich
5,Frank,52,Engineering,Hamburg
6,Grace,38,Marketing,Berlin
7,Hans,41,Sales,Cologne
8,Ingrid,27,Engineering,Munich
9,Jorge,33,Marketing,Hamburg


## HBase via happybase

In [5]:
import happybase

hbase_conn = happybase.Connection(host="hbase-master")
hbase_conn.open()

if b"jupyter_demo" in hbase_conn.tables():
    hbase_conn.disable_table("jupyter_demo")
    hbase_conn.delete_table("jupyter_demo")
hbase_conn.create_table("jupyter_demo", {"cf": dict()})

table = hbase_conn.table("jupyter_demo")
table.put(b"row1", {b"cf:greeting": b"hello from jupyter"})
table.put(b"row2", {b"cf:greeting": b"hbase is genuinely still alive"})

hbase_pd = pd.DataFrame(
    [{"row": row.decode(), **{k.decode(): v.decode() for k, v in data.items()}} for row, data in table.scan()]
)
hbase_pd

,row,cf:greeting
0,row1,hello from jupyter
1,row2,hbase is genuinely still alive


## Engine comparison

Same input, two engines, run earlier in this notebook, the actual thesis
of this whole project.

In [6]:
comparison = (
    spark_counts_pd.rename(columns={"count": "spark"})
    .merge(mrjob_counts_pd.rename(columns={"count": "mrjob"}), on="word", how="outer")
    .sort_values("word")
    .reset_index(drop=True)
)
comparison

,word,spark,mrjob
0,hadoop,2,2
1,hello,2,2
2,modern,1,1
3,world,1,1
